In [1]:
import numpy as np  
import pandas as pd

In [2]:
df=pd.read_csv('cleaned_nepali_houses_v3.csv')

In [3]:
temp_df=df.copy()

Clean-up complete!
Final residential row count: 2562
Remaining nulls in Bedrooms: 0


In [20]:


# --- STEP 1: Separate Commercial Suspects ---
df_res = df[df['is_commercial_suspect'] == False].copy()
df_comm = df[df['is_commercial_suspect'] == True].copy()

# --- STEP 2: Handle Price Outliers (Log-IQR) ---
df_res['log_price'] = np.log1p(df_res['total_price'])
Q1 = df_res['log_price'].quantile(0.25)
Q3 = df_res['log_price'].quantile(0.75)
IQR = Q3 - Q1
df_res = df_res[df_res['log_price'] >= Q1 - 1.5*IQR].copy()

# --- STEP 3: Handle land_area_aana Outliers ---
df_res['land_area_aana'] = df_res['land_area_aana'].clip(upper=30)

# --- STEP 4: Handle build_up_area Outliers ---
# only remove the lower fence outliers (likely data entry errors) and keep the upper end:
df_res = df_res[df_res['build_up_area'] <= 15000].copy()

# --- STEP 5: Segment-Aware Imputation ---
# 5a. First, recreate the segment column so the loop doesn't crash
q_labels = ['Budget', 'Mid-Range', 'High-End', 'Ultra-Luxury']
df_res['price_segment'] = pd.qcut(df_res['total_price'], q=4, labels=q_labels)

# 5b. Create the incomplete listing flag
df_res['is_incomplete_listing'] = (df_res['bedrooms'].isna() & df_res['bathrooms'].isna()).astype(int)

# 5c. Impute based on segments
for segment in q_labels:
    mask = df_res['price_segment'] == segment
    # Get medians for this specific segment
    bed_med = df_res.loc[mask, 'bedrooms'].median()
    bath_med = df_res.loc[mask, 'bathrooms'].median()
    
    # Fill the NaNs
    df_res.loc[mask & df_res['bedrooms'].isna(), 'bedrooms'] = bed_med
    df_res.loc[mask & df_res['bathrooms'].isna(), 'bathrooms'] = bath_med

print(f"✅ Success! Residential rows remaining: {len(df_res)}")
print(f"✅ Nulls remaining: {df_res['bedrooms'].isnull().sum()}")

✅ Success! Residential rows remaining: 2506
✅ Nulls remaining: 0


In [17]:
pd.set_option('display.max_columns',None)

In [18]:
df_res['price_segment'].value_counts()

price_segment
Budget          702
High-End        654
Ultra-Luxury    585
Mid-Range       565
Name: count, dtype: int64

In [19]:


removed = df[~df.index.isin(df_res.index)]
print(removed[['total_price','land_area_aana','district']].describe())

        total_price  land_area_aana
count  1.230000e+02       105.00000
mean   4.138780e+07        12.03819
std    5.724001e+07        15.69267
min    2.100000e+06         1.20000
25%    8.000000e+06         4.00000
50%    2.400000e+07         5.50000
75%    4.500000e+07        13.20000
max    4.000000e+08       103.70000


In [21]:
print(df_res.isnull().sum()[df_res.isnull().sum() > 0])

floors              4
facing             49
road_width_feet     5
built_year_ad      18
house_age          18
dtype: int64


In [22]:
# --- STEP: Handle Remaining Nulls ---

# 1. floors — fill with mode
df_res['floors'] = df_res['floors'].fillna(df_res['floors'].mode()[0])

# 2. facing — fill with mode
df_res['facing'] = df_res['facing'].fillna(df_res['facing'].mode()[0])

# 3. road_width_feet — fill with median
df_res['road_width_feet'] = df_res['road_width_feet'].fillna(df_res['road_width_feet'].median())

# 4. built_year_ad — segment-aware median, then derive house_age
for segment in df_res['price_segment'].unique():
    mask = df_res['price_segment'] == segment
    year_med = df_res.loc[mask, 'built_year_ad'].median()
    df_res.loc[mask & df_res['built_year_ad'].isna(), 'built_year_ad'] = year_med

# Derive house_age from filled built_year_ad to keep them in sync
df_res['house_age'] = 2025 - df_res['built_year_ad']

# --- VERIFY ---
print("=== Remaining Nulls ===")
remaining = df_res.isnull().sum()
print(remaining[remaining > 0] if remaining[remaining > 0].any() else "✅ No nulls remaining")

print("\n=== Math Check (built_year_ad vs house_age) ===")
print(df_res[['built_year_ad', 'house_age']].head(10))

print("\n=== Final Shape ===")
print(df_res.shape)

# --- SAVE ---
df_res.to_csv('housing_model_ready_after_outlier_treatment.csv', index=False)
df_comm.to_csv('housing_commercial_set.csv', index=False)
print("\n✅ Files saved successfully")

=== Remaining Nulls ===
✅ No nulls remaining

=== Math Check (built_year_ad vs house_age) ===
   built_year_ad  house_age
0         2019.0        6.0
1         2019.0        6.0
2         2003.0       22.0
3         2002.0       23.0
4         2017.0        8.0
5         2008.0       17.0
6         2009.0       16.0
7         2018.0        7.0
8         2022.0        3.0
9         2013.0       12.0

=== Final Shape ===
(2506, 33)

✅ Files saved successfully


In [23]:
df_res.shape

(2506, 33)

In [24]:
df_res.sample(12)

,category,title,district,neighborhood,location_raw,total_price,price_per_aana,land_area_aana,build_up_area,floors,facing,road_width_feet,bedrooms,bathrooms,parking_cars,parking_bikes,built_year_ad,house_age,is_under_construction,amenity_count,has_modular_kitchen,has_parquet,has_drainage,has_solar,has_parking,has_garden,is_wide_road,is_commercial_suspect,is_area_estimated,luxury_score,log_price,price_segment,is_incomplete_listing
509,house,"Attractive house for sale at Setipakha, Hattiban",Lalitpur,Setipakha,"Setipakha, Lalitpur",26500000.0,8.548387e+06,3.1,1856.71,2.5,North East,13.0,4.0,4.0,0.0,0.0,2007.0,18.0,False,4,0,0,1,0,1,0,False,False,True,2,17.092655,Mid-Range,0
1800,house,Beautiful spacious semi bunglow on sale at Pas...,Kathmandu,Pasikot,"Pasikot, Kathmandu",24000000.0,6.000000e+06,4.0,2000.00,2.5,South East,20.0,5.0,2.0,0.0,0.0,2017.0,8.0,False,11,1,0,1,0,1,1,False,False,False,4,16.993564,Budget,0
2578,house,KHUMALTAR House,Lalitpur,Khumaltar,"Khumaltar, Lalitpur",25000000.0,7.142857e+06,3.5,1905.00,2.5,South,16.0,6.0,3.0,1.0,3.0,2015.0,10.0,False,0,0,0,0,0,0,0,False,False,False,0,17.034386,Budget,0
2545,house,Sangam Semi-Commercial House,Kathmandu,Tarkeshwor,"Tarkeshwor, Kathmandu",36000000.0,1.125000e+07,3.2,3122.00,2.5,North West,20.0,6.0,4.0,1.0,3.0,2021.0,4.0,False,0,0,0,0,0,0,0,False,False,False,0,17.399030,High-End,0
2513,house,Kohinoor House for Sale,Kathmandu,Bafal,"Bafal, Kathmandu",40000000.0,9.523810e+06,4.2,2235.00,2.5,North,20.0,4.0,4.0,1.0,3.0,2008.0,17.0,False,4,1,0,0,0,0,0,False,False,False,1,17.504390,High-End,0
2171,house,"House on sale at Bhatkyapati, Kirtipur",Kathmandu,Bhatkepati,"Bhatkepati, Kathmandu",17500000.0,5.303030e+06,3.3,1581.19,2.0,South,13.0,5.0,2.0,0.0,0.0,2020.0,5.0,False,2,0,0,0,0,0,0,False,False,True,0,16.677711,Budget,0
630,house,"Residential house on sale at Panitanki, Tikath...",Lalitpur,Tikathali,"Tikathali, Lalitpur",27000000.0,6.585366e+06,4.1,2455.64,2.5,North West,13.0,7.0,4.0,0.0,0.0,2019.0,6.0,False,4,0,0,1,0,1,0,False,False,True,2,17.111347,Mid-Range,0
1540,house,Beautiful house for sale in Dhapasi,Kathmandu,Dhapasi,"Dhapasi, Kathmandu",44000000.0,6.285714e+06,7.0,2535.00,2.5,North East,13.0,5.0,4.0,0.0,0.0,2021.0,4.0,False,7,0,0,1,0,1,1,False,False,False,3,17.599700,High-End,0
2164,house,"House on sale in Thankot, near by High Vision ...",Kathmandu,Thankot,"Thankot, Kathmandu",11500000.0,3.709677e+06,3.1,742.68,1.0,South East,13.0,3.0,1.0,0.0,0.0,2019.0,6.0,False,4,0,0,1,0,1,0,False,False,True,2,16.257858,Budget,0
489,house,"Attractive 5 bhk house on sale at Imadol, Lali...",Lalitpur,Imadol,"Imadol, Lalitpur",27500000.0,9.166667e+06,3.0,1796.81,2.5,North West,20.0,5.0,3.0,0.0,0.0,2015.0,10.0,False,4,0,0,1,0,1,0,False,False,True,2,17.129697,Mid-Range,0
